# 07 - Chatbot de Planificación Dietética

Chatbot conversacional que guía al usuario a través de un proceso estructurado:

| Etapa | Descripción |
|-------|-------------|
| 1. Bienvenida | El bot se presenta y pregunta el objetivo |
| 2. Recopilación | Pregunta preferencias, restricciones, presupuesto, tiempo, personas |
| 3. Confirmación | Resume la info y pide confirmación |
| 4. Generación | Produce un `PlanComidas` validado con Pydantic |
| 5. Iteración | Ajusta el plan según feedback del usuario |

**Componentes clave:**
- `ChatMessageHistory` → Memoria de conversación
- `SystemMessage` + `ChatPromptTemplate` → Guía el flujo por etapas
- `llm.with_structured_output(PlanComidas)` → Salida validada

## Setup

In [ ]:
import os
import sys
from dotenv import load_dotenv

sys.path.insert(0, '../src')
load_dotenv(dotenv_path='../.env')

api_key = os.getenv('GOOGLE_API_KEY')
print('API key cargada:', 'Si' if api_key else 'No')

In [ ]:
from diet_planner import DietChatbot, PlanComidas

bot = DietChatbot(api_key=api_key)
print('DietChatbot listo')

## Etapa 1: Bienvenida

`start()` envía el primer mensaje y activa la presentación del bot.

In [ ]:
respuesta = bot.start()
print('NutriBot:', respuesta)

## Etapa 2: Recopilación de información

Cada llamada a `chat()` = 1 llamada a la API. El bot hace **una pregunta por turno**.

In [ ]:
# Turno 1: Objetivo
r = bot.chat('Quiero bajar de peso y comer más saludable')
print('NutriBot:', r)

In [ ]:
# Turno 2: Preferencias
r = bot.chat('Soy omnívoro, como de todo')
print('NutriBot:', r)

In [ ]:
# Turno 3: Restricciones
r = bot.chat('Soy intolerante a la lactosa, nada más')
print('NutriBot:', r)

In [ ]:
# Turno 4: Presupuesto
r = bot.chat('Tengo unos 80 dolares semanales para comida')
print('NutriBot:', r)

In [ ]:
# Turno 5: Tiempo
r = bot.chat('Puedo cocinar unos 30 minutos al día, máximo 45')
print('NutriBot:', r)

In [ ]:
# Turno 6: Personas
r = bot.chat('Solo para mí, 1 persona')
print('NutriBot:', r)

## Etapa 3: Confirmación

In [ ]:
# El usuario confirma la info recopilada
r = bot.chat('Sí, todo está correcto, por favor genera el plan')
print('NutriBot:', r)

## Etapa 4: Generar Plan

`generate_plan()` toma toda la conversación como contexto y usa `with_structured_output(PlanComidas)`
para garantizar una respuesta validada por Pydantic.

In [ ]:
print('Generando plan de comidas...')
plan = bot.generate_plan()

print(f'Tipo de retorno: {type(plan).__name__}')  # PlanComidas
print(f'Objetivo: {plan.objetivo}')
print(f'Dias generados: {len(plan.semana)}')
print(f'Costo estimado: ${plan.costo_total_estimado_usd:.2f} USD')

In [ ]:
# Ver el plan completo formateado
print(bot.format_plan())

In [ ]:
# Acceder a campos individuales del plan (es un objeto Pydantic)
lunes = plan.semana[0]
print(f'Lunes:')
print(f'  Desayuno: {lunes.desayuno.nombre}')
print(f'    Ingredientes: {", ".join(lunes.desayuno.ingredientes)}')
print(f'    Tiempo: {lunes.desayuno.tiempo_preparacion} min')
print()
print(f'  Almuerzo: {lunes.almuerzo.nombre}')
print(f'  Cena: {lunes.cena.nombre}')

## Etapa 5: Iterar con Feedback

El usuario puede pedir ajustes. `chat()` sigue funcionando con el historial completo.

In [ ]:
# Feedback sobre el plan
r = bot.chat('El plan está bien pero no me gustan los huevos, puedes cambiarlos por algo diferente?')
print('NutriBot:', r)

In [ ]:
# Regenerar el plan con el feedback incorporado en el historial
plan_v2 = bot.generate_plan()
print(bot.format_plan())

## Inspeccionar el Historial de Conversación

In [ ]:
print(f'Total de turnos en la conversacion: {len(bot.history)}')
print(f'Turnos del usuario: {sum(1 for r, _ in bot.history if r == "human")}')
print(f'Turnos del bot: {sum(1 for r, _ in bot.history if r == "ai")}')
print()
print('Ultimos 2 turnos:')
for role, content in bot.history[-2:]:
    label = 'Usuario' if role == 'human' else 'NutriBot'
    print(f'  [{label}]: {content[:120]}...')

## Resumen: Arquitectura del Chatbot

```
DietChatbot
│
├── llm (ChatGoogleGenerativeAI)          → Conversación libre
├── structured_llm (with_structured_output) → Genera PlanComidas validado
├── history [(role, content), ...]        → Memoria conversacional
│
├── start()           → ETAPA 1: Activa la bienvenida
├── chat(input)       → ETAPA 2-3-5: Conversación libre con memoria
├── generate_plan()   → ETAPA 4: Structured output desde conversación
├── format_plan()     → Formatea PlanComidas como texto
└── reset()           → Reinicia historial y plan

Flujo de mensajes en chat():
  history → SystemMessage + HumanMessages + AIMessages → llm.invoke() → AIMessage

Flujo en generate_plan():
  history → texto plano → ChatPromptTemplate → structured_llm.invoke() → PlanComidas
```

In [ ]:
# Reiniciar para empezar una nueva conversación
bot.reset()
print('Chatbot reiniciado')
print(f'Historial: {len(bot.history)} turnos')
print(f'Plan: {bot.plan}')